### 1 Load the processed data

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

user_mapping = pd.read_csv(PROCESSED_DIR / "user_mapping.csv")
movie_mapping = pd.read_csv(PROCESSED_DIR / "movie_mapping.csv")
movie_popularity = pd.read_csv(PROCESSED_DIR / "movie_popularity.csv")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(train_df.head())


Training: (80419, 6)
Validation: (9818, 6)
Test: (10599, 6)


,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,804,4.0,2000-07-30 18:08:19+00:00,0,632
1,1,1210,5.0,2000-07-30 18:08:19+00:00,0,911
2,1,2018,5.0,2000-07-30 18:08:43+00:00,0,1493
3,1,2628,4.0,2000-07-30 18:08:43+00:00,0,1979
4,1,2826,4.0,2000-07-30 18:08:43+00:00,0,2126


Build a user-item interaction matrix

In [2]:

from scipy.sparse import csr_matrix

NUM_USERS = len(user_mapping)
NUM_MOVIES = len(movie_mapping)

user_item_matrix = csr_matrix(
    (
        train_df["rating"].values,
        (
            train_df["user_idx"].values,
            train_df["movie_idx"].values
        )
    ),
    shape=(NUM_USERS, NUM_MOVIES)
)

print("Matrix Shape:", user_item_matrix.shape)
print("Non-zero Interactions:", user_item_matrix.nnz)

print(
    "Matrix Sparsity:",
    round(
        100 * (1 - user_item_matrix.nnz / (NUM_USERS * NUM_MOVIES)),
        2
    ),
    "%"
)


Matrix Shape: (610, 9742)
Non-zero Interactions: 80419
Matrix Sparsity: 98.65 %


Popularity-based recommender

In [3]:

popular_movies = (
    train_df.groupby("movie_idx")
    .agg(
        rating_count=("rating", "count"),
        average_rating=("rating", "mean")
    )
    .reset_index()
)

popular_movies = popular_movies.sort_values(
    ["rating_count", "average_rating"],
    ascending=False
)

popular_movie_ids = popular_movies["movie_idx"].tolist()

print("Top 10 Popular Movies:")
display(
    popular_movies.head(10).merge(
        movie_mapping,
        on="movie_idx",
        how="left"
    ).merge(
        movies[["movieId", "title"]],
        on="movieId",
        how="left"
    ) if "movies" in globals() else popular_movies.head(10)
)


Top 10 Popular Movies:


,movie_idx,rating_count,average_rating
302,314,303,4.156766
266,277,281,4.416370
247,257,271,4.175277
1864,1939,259,4.193050
489,510,251,4.163347
215,224,236,4.220339
96,97,224,4.017857
402,418,222,3.761261
0,0,201,3.915423
486,507,200,3.945000


User-Based Collaborative Filtering

In [4]:

from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_item_matrix)

print("User Similarity Matrix:", user_similarity.shape)


User Similarity Matrix: (610, 610)


Generate personalized recommendations

In [6]:

def recommend_user_based(user_idx, top_k=10, n_neighbors=20):

    similarities = user_similarity[user_idx].copy()

    similarities[user_idx] = 0

    neighbor_indices = np.argsort(similarities)[::-1][:n_neighbors]

    neighbor_weights = similarities[neighbor_indices]

    neighbor_ratings = user_item_matrix[neighbor_indices]

    scores = np.asarray(
        neighbor_weights @ neighbor_ratings
    ).flatten()

    seen_movies = user_item_matrix[user_idx].indices

    scores[seen_movies] = -np.inf

    recommended_indices = np.argsort(scores)[::-1][:top_k]

    return recommended_indices.tolist()


sample_user = 0

recommendations = recommend_user_based(sample_user, top_k=10)

print("Recommended Movie Indices:")
print(recommendations)


Recommended Movie Indices:
[507, 793, 902, 939, 1067, 964, 1211, 969, 659, 474]


Evaluate Recall@10 and NDCG@10

In [10]:

def recall_at_k(recommended, relevant, k=10):

    recommended = recommended[:k]

    if len(relevant) == 0:
        return np.nan

    hits = len(set(recommended) & set(relevant))

    return hits / len(relevant)


def ndcg_at_k(recommended, relevant, k=10):

    recommended = recommended[:k]

    relevant = set(relevant)

    dcg = sum(
        1 / np.log2(rank + 2)
        for rank, item in enumerate(recommended)
        if item in relevant
    )

    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / np.log2(rank + 2)
        for rank in range(ideal_hits)
    )

    return dcg / idcg if idcg > 0 else 0


Evaluate popularity baseline

In [11]:

POSITIVE_THRESHOLD = 4.0
K = 10

test_relevant = (
    test_df[test_df["rating"] >= POSITIVE_THRESHOLD]
    .groupby("user_idx")["movie_idx"]
    .apply(set)
    .to_dict()
)

train_seen = (
    train_df.groupby("user_idx")["movie_idx"]
    .apply(set)
    .to_dict()
)


def recommend_popularity(user_idx, top_k=10):

    seen = train_seen.get(user_idx, set())

    candidates = [
        movie_idx
        for movie_idx in popular_movie_ids
        if movie_idx not in seen
    ]

    return candidates[:top_k]


popularity_recalls = []
popularity_ndcgs = []

for user_idx, relevant_items in test_relevant.items():

    recommendations = recommend_popularity(user_idx, K)

    popularity_recalls.append(
        recall_at_k(recommendations, relevant_items, K)
    )

    popularity_ndcgs.append(
        ndcg_at_k(recommendations, relevant_items, K)
    )


print("Popularity Recall@10:", np.nanmean(popularity_recalls))
print("Popularity NDCG@10:", np.nanmean(popularity_ndcgs))


Popularity Recall@10: 0.04263400460062029
Popularity NDCG@10: 0.04384374003108282


Evaluate user-based collaborative filtering

In [12]:

user_cf_recalls = []
user_cf_ndcgs = []

for user_idx, relevant_items in test_relevant.items():

    recommendations = recommend_user_based(
        user_idx,
        top_k=K,
        n_neighbors=20
    )

    user_cf_recalls.append(
        recall_at_k(recommendations, relevant_items, K)
    )

    user_cf_ndcgs.append(
        ndcg_at_k(recommendations, relevant_items, K)
    )


print("User CF Recall@10:", np.nanmean(user_cf_recalls))
print("User CF NDCG@10:", np.nanmean(user_cf_ndcgs))


User CF Recall@10: 0.06656902528012659
User CF NDCG@10: 0.06371726776585729


Compare the baseline models

In [13]:

baseline_results = pd.DataFrame({
    "Model": [
        "Popularity-Based",
        "User-Based Collaborative Filtering"
    ],
    "Recall@10": [
        np.nanmean(popularity_recalls),
        np.nanmean(user_cf_recalls)
    ],
    "NDCG@10": [
        np.nanmean(popularity_ndcgs),
        np.nanmean(user_cf_ndcgs)
    ]
})

display(baseline_results)


,Model,Recall@10,NDCG@10
0,Popularity-Based,0.042634,0.043844
1,User-Based Collaborative Filtering,0.066569,0.063717


Result
- Popularity-Based: Recommends movies popular among users generally. It does not personalize recommendations.
- User-Based Collaborative Filtering: Finds users with similar rating histories and recommends movies they liked.
- Collaborative Filtering has improved Recall@10 by approximately 56% over the popularity baseline.